# VLM2Vec V2 — LongDocURL Fine-tuning

**流程：**

1. 挂载 Google Drive（数据在 Drive 里）
2. Clone VLM2Vec 并安装依赖
3. 上传 / 注册自定义 dataset loader
4. 写入配置文件
5. 启动训练

> 确保 Runtime → Change runtime type 已选 **A100 GPU**


## Cell 0 — 检查 GPU


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

## Cell 1 — 挂载 Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# ── 修改为你的实际路径 ──────────────────────────────────────────
import os

DRIVE_BASE = "/content/drive/MyDrive/vlm2vec_longdocurl"  # Drive 根目录
TRIPLETS_TRAIN = f"{DRIVE_BASE}/data/triplets/triplets_train.jsonl"
TRIPLETS_VAL = f"{DRIVE_BASE}/data/triplets/triplets_val.jsonl"
PAGE_IMAGES_DIR = f"{DRIVE_BASE}/data/triplets/page_images"  # PNG 渲染目录
OUTPUT_DIR = f"{DRIVE_BASE}/outputs/longdocurl_vlm2vec"

# 验证路径
for p in [TRIPLETS_TRAIN, TRIPLETS_VAL, PAGE_IMAGES_DIR]:
    exists = os.path.exists(p)
    print(f'  {"✓" if exists else "✗"} {p}')

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Cell 2 — Clone VLM2Vec & 安装依赖


In [ ]:
import os
if not os.path.exists('/content/VLM2Vec'):
    !git clone https://github.com/TIGER-AI-Lab/VLM2Vec /content/VLM2Vec
else:
    print('VLM2Vec already cloned, skipping.')

%cd /content/VLM2Vec

In [ ]:
# 安装依赖（约 3~5 分钟）
!pip install -q -r requirements.txt
!pip install -q qwen-vl-utils
!pip install -q flash-attn --no-build-isolation
print('Done.')

## Cell 3 — 注册自定义 Dataset Loader


In [ ]:
loader_code = """
from __future__ import annotations
import json
from pathlib import Path
from datasets import Dataset
from src.data.dataset.base_pair_dataset import (
    AutoPairDataset, add_metainfo_hook, MULTIMODAL_FEATURES, RESOLUTION_MAPPING,
)
from src.model.processor import VLM_IMAGE_TOKENS

QUERY_PROMPT = (
    "This query is related to retrieving a relevant page from a multi-page document. "
    "The retrieved page should contain text, tables, figures, or structured information "
    "necessary to answer the query: "
)
PAGE_PROMPT = "A page from a document containing text, tables, or figures."

@add_metainfo_hook
def data_prepare(batch_dict, *args, **kwargs):
    model_backbone  = kwargs["model_backbone"]
    image_resolution = kwargs["image_resolution"]
    image_token = VLM_IMAGE_TOKENS[model_backbone]
    resolution  = RESOLUTION_MAPPING.get(image_resolution, None)

    query_texts, query_images = [], []
    pos_texts,   pos_images   = [], []
    neg_texts,   neg_images   = [], []

    for question, pos_path, neg_paths in zip(
        batch_dict["question"],
        batch_dict["pos_image_path"],
        batch_dict["neg_image_paths"],
    ):
        query_texts.append(f"{QUERY_PROMPT}{question}")
        query_images.append({"bytes": [None], "paths": [None], "resolutions": [[224, 224]]})

        pos_bytes = Path(pos_path).read_bytes() if pos_path and Path(pos_path).exists() else b""
        pos_texts.append(f"{PAGE_PROMPT} {image_token}")
        pos_images.append({"bytes": [pos_bytes], "paths": [pos_path], "resolutions": [resolution or [224, 224]]})

        neg_t, neg_i = [], []
        for neg_path in neg_paths:
            neg_bytes = Path(neg_path).read_bytes() if neg_path and Path(neg_path).exists() else b""
            neg_t.append(f"{PAGE_PROMPT} {image_token}")
            neg_i.append({"bytes": [neg_bytes], "paths": [neg_path], "resolutions": [resolution or [224, 224]]})
        if not neg_t:
            neg_t.append("")
            neg_i.append({"bytes": [b""], "paths": [""], "resolutions": [[224, 224]]})
        neg_texts.append(neg_t)
        neg_images.append(neg_i)

    return {"query_text": query_texts, "query_image": query_images,
            "pos_text": pos_texts, "pos_image": pos_images,
            "neg_text": neg_texts, "neg_image": neg_images}


DATASET_PARSER_NAME = "longdocurl"

@AutoPairDataset.register(DATASET_PARSER_NAME)
def load_longdocurl_dataset(model_args, data_args, training_args, *args, **kwargs):
    dataset_path = kwargs.get("dataset_path")
    if not dataset_path:
        raise ValueError("longdocurl loader requires `dataset_path` in yaml config")
    records = []
    with open(dataset_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            neg_paths = [c["image_path"] for c in r.get("hard_neg_chunks", []) if c.get("image_path")]
            records.append({"question": r["question"], "answer": r["answer"],
                            "pos_image_path": r["pos_chunk"]["image_path"],
                            "neg_image_paths": neg_paths})
    num_sample = kwargs.get("num_sample_per_subset", getattr(data_args, "num_sample_per_subset", None))
    if num_sample and num_sample < len(records):
        records = records[:int(num_sample)]
    num_rows = len(records)
    dataset = Dataset.from_list(records)
    num_shards = training_args.dataloader_num_workers if training_args.dataloader_num_workers > 0 else 1
    dataset = dataset.to_iterable_dataset(num_shards=num_shards)
    kwargs["model_backbone"]      = model_args.model_backbone
    kwargs["image_resolution"]    = data_args.image_resolution
    kwargs["global_dataset_name"] = kwargs.get("global_dataset_name", f"longdocurl/{Path(dataset_path).stem}")
    dataset = dataset.map(lambda x: data_prepare(x, **kwargs),
                          batched=True, batch_size=32,
                          remove_columns=["question", "answer", "pos_image_path", "neg_image_paths"],
                          drop_last_batch=True, features=MULTIMODAL_FEATURES)
    setattr(dataset, "num_rows", num_rows)
    return dataset
"""

loader_path = "/content/VLM2Vec/src/data/dataset/longdocurl_dataset.py"
with open(loader_path, "w") as f:
    f.write(loader_code)

# __init__.py 里注册
init_path = "/content/VLM2Vec/src/data/dataset/__init__.py"
init_text = open(init_path).read()
import_line = "from src.data.dataset.longdocurl_dataset import *\n"
if import_line not in init_text:
    with open(init_path, "a") as f:
        f.write(import_line)
    print("✓ Loader registered in __init__.py")
else:
    print("✓ Loader already registered")

## Cell 4 — 写入配置文件


In [ ]:
import os, yaml

EXP_DIR = "/content/VLM2Vec/experiments/longdocurl"
os.makedirs(EXP_DIR, exist_ok=True)

# ── 数据配置 ────────────────────────────────────────────────────
data_config = {
    "longdocurl_train": {
        "dataset_parser": "longdocurl",
        "dataset_path": TRIPLETS_TRAIN,
        "weight": 1,
    }
}
with open(f"{EXP_DIR}/data_config.yaml", "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)
print("✓ data_config.yaml written")

# ── 训练超参 ────────────────────────────────────────────────────
# 可以在这里直接修改超参
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"  # 换 7B: 'Qwen/Qwen2-VL-7B-Instruct'
LORA_R = 16
LORA_ALPHA = 64
BATCH_SIZE = 4
GRAD_ACCUM = 4  # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR = 2e-5
EPOCHS = 3
MAX_PIXELS = 501760  # 28*28*640，减小可节省显存

print(f"Model:          {MODEL_NAME}")
print(f"LoRA r/alpha:   {LORA_R}/{LORA_ALPHA}")
print(f"Effective batch:{BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning rate:  {LR}")
print(f"Epochs:         {EPOCHS}")
print(f"Output dir:     {OUTPUT_DIR}")

## Cell 5 — 启动训练


In [ ]:
import subprocess, sys

%cd /content/VLM2Vec

cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    '--nproc_per_node', '1',
    'train.py',
    # ── Model ────────────────────────────────────────────────
    '--model_name',            MODEL_NAME,
    '--model_backbone',        'qwen2_vl',
    '--pooling',               'last',
    '--normalize',             'true',
    '--lora',                  'true',
    '--lora_r',                str(LORA_R),
    '--lora_alpha',            str(LORA_ALPHA),
    '--lora_dropout',          '0.05',
    '--lora_target_modules',   'q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj',
    # ── Data ─────────────────────────────────────────────────
    '--dataset_config',        f'{EXP_DIR}/data_config.yaml',
    '--resize_use_processor',  'true',
    '--resize_min_pixels',     str(28*28*4),
    '--resize_max_pixels',     str(MAX_PIXELS),
    '--num_hardneg',           '3',
    # ── Training ─────────────────────────────────────────────
    '--output_dir',            OUTPUT_DIR,
    '--num_train_epochs',      str(EPOCHS),
    '--per_device_train_batch_size', str(BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRAD_ACCUM),
    '--learning_rate',         str(LR),
    '--warmup_ratio',          '0.05',
    '--lr_scheduler_type',     'cosine',
    '--bf16',                  'true',
    '--tf32',                  'true',
    '--dataloader_num_workers','2',
    '--logging_steps',         '10',
    '--save_steps',            '200',
    '--save_total_limit',      '3',
    '--image_encoder_freeze',  'true',
    '--grad_cache',            'true',
    '--gc_q_chunk_size',       '2',
    '--gc_p_chunk_size',       '2',
    '--interleave_stopping_strategy', 'first_exhausted',
    '--remove_unused_columns', 'false',
    '--resume_from',           'none',
    '--report_to',             'none',
]

print('Launching training...')
print('CMD:', ' '.join(cmd))
result = subprocess.run(cmd)
print('Return code:', result.returncode)

## Cell 6 — 验证 checkpoint 输出


In [ ]:
import os

print("Output dir contents:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

## Cell 7 — 快速推理验证（可选）


In [ ]:
import sys

sys.path.insert(0, "/content/VLM2Vec")

import torch
from PIL import Image
from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN2_VL, VLM_IMAGE_TOKENS
from src.model.vlm_backbone.qwen2_vl.qwen_vl_utils import process_vision_info

# 找最新 checkpoint
import os

ckpts = sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")])
latest_ckpt = os.path.join(OUTPUT_DIR, ckpts[-1]) if ckpts else OUTPUT_DIR
print("Loading checkpoint:", latest_ckpt)

model_args = ModelArguments(
    model_name=MODEL_NAME,
    checkpoint_path=latest_ckpt,
    pooling="last",
    normalize=True,
    model_backbone="qwen2_vl",
    lora=True,
)
data_args = DataArguments()
processor = load_processor(model_args, data_args)
model = MMEBModel.load(model_args).to("cuda", dtype=torch.bfloat16).eval()
print("Model loaded.")

In [ ]:
import json
import numpy as np

# 从 val 集取几条做快速 Recall@1 验证
val_records = []
with open(TRIPLETS_VAL) as f:
    for line in f:
        val_records.append(json.loads(line.strip()))

sample = val_records[:20]  # 只取前20条
image_token = VLM_IMAGE_TOKENS[QWEN2_VL]
PAGE_PROMPT = "A page from a document containing text, tables, or figures."
QUERY_PROMPT = (
    "This query is related to retrieving a relevant page from a multi-page document. "
    "The retrieved page should contain text, tables, figures, or structured information "
    "necessary to answer the query: "
)


def encode_text(text):
    inputs = processor(text=text, return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        return model(qry=inputs)["qry_reps"].cpu().float().numpy()


def encode_image(image_path, text):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(text=f"{image_token} {text}", images=[img], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        return model(tgt=inputs)["tgt_reps"].cpu().float().numpy()


# 计算 Recall@1
hits = 0
for i, r in enumerate(sample):
    q_emb = encode_text(f"{QUERY_PROMPT}{r['question']}")
    pos_emb = encode_image(r["pos_chunk"]["image_path"], PAGE_PROMPT)
    neg_embs = [
        encode_image(c["image_path"], PAGE_PROMPT)
        for c in r["hard_neg_chunks"]
        if c.get("image_path")
    ]

    all_embs = np.vstack([pos_emb] + neg_embs)  # pos at index 0
    scores = (q_emb @ all_embs.T).squeeze()
    if scores.argmax() == 0:
        hits += 1

print(
    f"Recall@1 on {len(sample)} val samples: {hits}/{len(sample)} = {hits/len(sample):.2%}"
)